# Autoencoder V2 — Quality-Focused CASIA Training on Colab

This notebook fine-tunes the **existing validated RTX80 convolutional Autoencoder**; it does not change the architecture, 24× bottleneck, dataset split, or `[0,1]` preprocessing.

The run initializes from `best_autoencoder_rtx80_portable.pth` (epoch 76), then fine-tunes with a lower learning rate. New artifacts use separate `*_rtx80_finetuned` paths, so both the RTX80 baseline and the earlier V2 run remain recoverable. Better results are selected strictly by validation MSE—improvement is targeted, not guaranteed.

## 1. Colab setup
Select **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
%pip install -q kagglehub Pillow matplotlib numpy pandas scikit-image scikit-learn

In [ ]:
import os, sys, json, csv, shutil, subprocess, importlib
from pathlib import Path, PureWindowsPath
import torch

assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime > Change runtime type > T4 GPU"
GPU_NAME = torch.cuda.get_device_name(0)
print("CUDA:", torch.cuda.is_available())
print("GPU:", GPU_NAME)

## 2. Obtain project code
The CASIA images and trained weights are not uploaded to GitHub.

In [ ]:
REPO_URL = "https://github.com/chetanraje27/Digital-Evidence-GenAI.git"
PROJECT_ROOT = Path("/content/Digital-Evidence-GenAI")
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Project:", PROJECT_ROOT)

## 3. Download CASIA v2.0 and validate paths
The committed manifests remain the source of truth. Ground-truth PNG masks are excluded.

In [ ]:
import kagglehub

RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
downloaded = None
if not (RAW_DIR / "CASIA2" / "Au").is_dir():
    downloaded = Path(kagglehub.dataset_download(
        "divg07/casia-20-image-tampering-detection-dataset", output_dir=str(RAW_DIR)
    )).resolve()
    print("Downloaded to:", downloaded)

# KaggleHub may ignore output_dir when it reuses Colab's mounted cache and
# return /kaggle/input/... instead. Search the returned path, not only RAW_DIR.
if not (RAW_DIR / "CASIA2").is_dir():
    search_roots = [p for p in (downloaded, RAW_DIR) if p is not None and p.exists()]
    candidates = []
    for root in search_roots:
        # Handle both a returned CASIA2 directory and a parent containing it.
        if root.name.casefold() == "casia2" and (root / "Au").is_dir() and (root / "Tp").is_dir():
            candidates.append(root)
        candidates.extend(
            p for p in root.rglob("CASIA2")
            if (p / "Au").is_dir() and (p / "Tp").is_dir()
        )
    assert candidates, (
        f"CASIA2/Au and CASIA2/Tp were not found under: {search_roots}. "
        "Inspect the printed Kaggle dataset location."
    )
    source_casia2 = candidates[0].resolve()
    os.symlink(source_casia2, RAW_DIR / "CASIA2", target_is_directory=True)
    print("Linked:", RAW_DIR / "CASIA2", "->", source_casia2)

SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
rows_by_split = {}
for name in ("train", "validation", "test"):
    with (SPLITS_DIR / f"{name}.csv").open(newline="", encoding="utf-8") as f:
        rows_by_split[name] = list(csv.DictReader(f))
all_rows = sum(rows_by_split.values(), [])
paths = [row["image_path"] for row in all_rows]
assert {name: len(rows) for name, rows in rows_by_split.items()} == {"train": 8830, "validation": 1892, "test": 1892}
assert len(paths) == len(set(p.casefold() for p in paths)) == 12614
assert all(not Path(p).is_absolute() and not PureWindowsPath(p).is_absolute() for p in paths)
assert all(Path(p).suffix.lower() != ".png" and "groundtruth" not in p.lower() for p in paths)
missing = [p for p in paths if not (PROJECT_ROOT / p).is_file()]
assert not missing, f"Missing images, first examples: {missing[:5]}"
print("Split and leakage validation: PASS", {k: len(v) for k, v in rows_by_split.items()})

## 4. Validate pipeline and architecture

In [ ]:
from ae_dataset import create_ae_dataloaders
from autoencoder import ConvolutionalAutoencoder

loaders = create_ae_dataloaders(SPLITS_DIR, image_size=128, batch_size=32, num_workers=2, seed=42)
batch = next(iter(loaders["train"]))["image"]
model = ConvolutionalAutoencoder()
with torch.inference_mode():
    latent = model.encode(batch[:2])
    output = model(batch[:2])
assert batch.shape == (32, 3, 128, 128)
assert latent.shape == (2, 32, 8, 8) and output.shape == (2, 3, 128, 128)
assert 0 <= batch.min() <= batch.max() <= 1
print("Batch:", tuple(batch.shape), "range:", (float(batch.min()), float(batch.max())))
print("Latent:", tuple(latent.shape), "parameters:", sum(p.numel() for p in model.parameters()), "compression: 24x")

## 5. Baseline checkpoint

Recommended: place `best_autoencoder_rtx80_portable.pth` in Google Drive at `/content/drive/MyDrive/Digital_Evidence/checkpoints/`. The repository copy is used as a fallback. The notebook stops if the validated RTX80 baseline is unavailable rather than silently starting a scientifically different scratch run.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_PROJECT = Path("/content/drive/MyDrive/Digital_Evidence")
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
baseline_candidates = [
    DRIVE_PROJECT / "checkpoints" / "best_autoencoder_rtx80_portable.pth",
    PROJECT_ROOT / "checkpoints" / "best_autoencoder_rtx80_portable.pth",
]
BASELINE_CHECKPOINT = next((p for p in baseline_candidates if p.is_file()), None)
assert BASELINE_CHECKPOINT is not None, (
    "RTX80 baseline checkpoint is unavailable. Put best_autoencoder_rtx80_portable.pth "
    "in the Drive checkpoints folder or pull the repository's LFS checkpoint bytes."
)
with BASELINE_CHECKPOINT.open("rb") as checkpoint_file:
    is_lfs_pointer = checkpoint_file.read(128).startswith(
        b"version https://git-lfs.github.com/spec/v1"
    )
assert not is_lfs_pointer, (
    "The RTX80 checkpoint is only a Git LFS pointer. In a Colab terminal run: "
    "git -C /content/Digital-Evidence-GenAI lfs pull --include="
    "checkpoints/best_autoencoder_rtx80_portable.pth"
)
ckpt = torch.load(BASELINE_CHECKPOINT, map_location="cpu", weights_only=True)
assert "model_state_dict" in ckpt, "RTX80 checkpoint has an unexpected format."
print("Baseline checkpoint:", BASELINE_CHECKPOINT)
print("Baseline epoch:", ckpt.get("epoch"), "validation MSE:", ckpt.get("validation_loss"))

## 6. Quality-focused configuration

- Existing architecture, 128×128 input, batch size 32, Adam, and MSE remain unchanged.
- Fine-tuning starts conservatively at `2e-5`; `ReduceLROnPlateau` can reduce it to `1e-7`.
- Up to 40 additional epochs; early stopping patience 8.
- CUDA mixed precision improves T4 speed without changing the model.
- The epoch-76 RTX80 weights remain selected unless validation MSE genuinely improves.

In [ ]:
from argparse import Namespace

V2_ROOT = PROJECT_ROOT
common = dict(
    splits_dir=SPLITS_DIR, image_size=128, batch_size=32, num_workers=2,
    learning_rate=2e-5,
    min_learning_rate=1e-7, weight_decay=1e-6, max_epochs=40,
    patience=8, lr_patience=2, lr_factor=0.5, min_delta=1e-7,
    seed=42, initial_checkpoint=BASELINE_CHECKPOINT,
)
TRAIN_ARGS = Namespace(**common,
    checkpoint_path=V2_ROOT / "checkpoints" / "best_autoencoder_rtx80_finetuned.pth",
    history_path=V2_ROOT / "results" / "ae_rtx80_finetuned_training_history.csv",
    summary_path=V2_ROOT / "results" / "ae_rtx80_finetuned_training_summary.json",
    curve_path=V2_ROOT / "outputs" / "ae_rtx80_finetuned" / "training_curve.png",
    grid_path=V2_ROOT / "outputs" / "ae_rtx80_finetuned" / "reconstruction_grid.png",
    smoke_test=False, smoke_batches=2, require_cuda=True,
)
print(vars(TRAIN_ARGS))

## 7. Required smoke test
This uses two train/validation batches and writes temporary artifacts only.

In [ ]:
import train_autoencoder_v2
importlib.reload(train_autoencoder_v2)

smoke_dir = Path("/content/ae_v2_smoke")
SMOKE_ARGS = Namespace(**{**vars(TRAIN_ARGS),
    "checkpoint_path": smoke_dir / "smoke.pth", "history_path": smoke_dir / "history.csv",
    "summary_path": smoke_dir / "summary.json", "curve_path": smoke_dir / "curve.png",
    "grid_path": smoke_dir / "grid.png", "smoke_test": True, "require_cuda": True,
})
smoke_summary = train_autoencoder_v2.train(SMOKE_ARGS)
assert smoke_summary["epochs_completed"] == 1
print("Smoke test: PASS", smoke_summary)

## 8. Full GPU training
Set `RUN_FULL_TRAINING = True`. The best validation checkpoint is saved, not merely the final epoch.

In [ ]:
RUN_FULL_TRAINING = True
if RUN_FULL_TRAINING:
    training_summary = train_autoencoder_v2.train(TRAIN_ARGS)
    print(json.dumps(training_summary, indent=2))
else:
    print("Full training skipped.")

## 9. Training curves and best checkpoint

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as DisplayImage

history = pd.read_csv(TRAIN_ARGS.history_path)
display(history.tail())
display(DisplayImage(filename=str(TRAIN_ARGS.curve_path)))
display(DisplayImage(filename=str(TRAIN_ARGS.grid_path)))
best = torch.load(TRAIN_ARGS.checkpoint_path, map_location="cpu", weights_only=True)
print("Best fine-tuning epoch:", best.get("epoch"))
print("Best validation MSE:", best.get("validation_loss"))

## 10. Complete held-out test evaluation
Evaluation uses all 1,892 test images and reports per-image MSE, PSNR, and SSIM.

In [ ]:
from evaluate_autoencoder import evaluate

EVAL_ARGS = Namespace(
    splits_dir=SPLITS_DIR, checkpoint_path=TRAIN_ARGS.checkpoint_path,
    per_image_csv=PROJECT_ROOT / "results" / "ae_rtx80_finetuned_test_per_image_metrics.csv",
    metrics_json=PROJECT_ROOT / "results" / "ae_rtx80_finetuned_test_metrics.json",
    reconstruction_grid=PROJECT_ROOT / "outputs" / "ae_rtx80_finetuned" / "test_reconstruction_grid.png",
    mse_plot=PROJECT_ROOT / "outputs" / "ae_rtx80_finetuned" / "authentic_vs_tampered_mse.png",
    ssim_plot=PROJECT_ROOT / "outputs" / "ae_rtx80_finetuned" / "authentic_vs_tampered_ssim.png",
    image_size=128, batch_size=32, num_workers=2, samples_per_class=3, seed=42,
)
test_metrics = evaluate(EVAL_ARGS)
print(json.dumps(test_metrics, indent=2))

## 11. Visual test results and baseline comparison

In [ ]:
display(DisplayImage(filename=str(EVAL_ARGS.reconstruction_grid)))
display(DisplayImage(filename=str(EVAL_ARGS.mse_plot)))

baseline_metrics_path = PROJECT_ROOT / "results" / "ae_rtx80_test_metrics.json"
if baseline_metrics_path.is_file():
    baseline_metrics = json.loads(baseline_metrics_path.read_text())
    comparison = pd.DataFrame([
        {"model": "RTX80 baseline", **{m: baseline_metrics["overall"][f"{m}_mean"] for m in ("mse", "psnr", "ssim")}},
        {"model": "New AE V2", **{m: test_metrics["overall"][f"{m}_mean"] for m in ("mse", "psnr", "ssim")}},
    ])
    display(comparison)
    improved = comparison.loc[1, "mse"] < comparison.loc[0, "mse"]
    print("V2 improved held-out test MSE:", improved)
    if not improved:
        print("Keep the RTX80 baseline as the active model; do not claim V2 is better.")
else:
    print("Baseline metrics file unavailable; V2 metrics are shown above.")
print("Note: authentic/tampered reconstruction differences are exploratory; the AE is not a forgery detector.")

## 12. Persist V2 artifacts to Google Drive

In [ ]:
artifact_paths = [
    TRAIN_ARGS.checkpoint_path, TRAIN_ARGS.history_path, TRAIN_ARGS.summary_path,
    TRAIN_ARGS.curve_path, TRAIN_ARGS.grid_path, EVAL_ARGS.per_image_csv,
    EVAL_ARGS.metrics_json, EVAL_ARGS.reconstruction_grid, EVAL_ARGS.mse_plot, EVAL_ARGS.ssim_plot,
]
for source in artifact_paths:
    if not source.is_file():
        raise FileNotFoundError(f"Required artifact was not produced: {source}")
    relative = source.relative_to(PROJECT_ROOT)
    destination = DRIVE_PROJECT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    print("Saved:", destination)

## 13. Final report

In [ ]:
print("GPU:", GPU_NAME)
print("Epochs completed:", training_summary["epochs_completed"])
print("Best fine-tuning epoch:", training_summary["best_epoch"])
print("Best validation MSE:", training_summary["best_validation_loss"])
print("Test MSE:", test_metrics["overall"]["mse_mean"])
print("Test PSNR:", test_metrics["overall"]["psnr_mean"])
print("Test SSIM:", test_metrics["overall"]["ssim_mean"])
print("Early stopping:", training_summary["early_stopping_triggered"])
print("Training seconds:", training_summary["training_time_seconds"])
print("Checkpoint:", TRAIN_ARGS.checkpoint_path)